In [2]:
import json
import re
from pathlib import Path

stats = []
for path in sorted(Path("../data/processed").glob("*.json")):
	doc = json.loads(path.read_text(encoding="utf-8"))
	tables = [b for b in doc["blocks"] if b["type"] == "table"]

	# header line has no text
	no_header = sum(not re.search(r"\w", t["content"].splitlines()[0]) for t in tables)
	one_row = sum(len(t["content"].splitlines()) <= 3 for t in tables)   # header + separator + 1 row
	no_context = sum(not t["context"] for t in tables)

	stats.append((doc["ticker"], len(tables), no_header, one_row, no_context))

print(f"{'ticker':<6} {'tables':>6} {'no_hdr':>7} {'1row':>6} {'no_ctx':>6}")
for ticker, n, nh, one, nc in sorted(stats, key=lambda s: -s[2] / s[1]):
	print(f"{ticker:<6} {n:>6} {nh / n:>7.0%} {one / n:>6.0%} {nc:>6}")

ticker tables  no_hdr   1row no_ctx
KO        114      7%     4%      0
AAPL       59      7%     2%      0
CAT       119      7%     0%      0
NKE        75      7%     8%      0
V          90      6%    10%      0
GOOGL      81      5%     4%      0
PFE       119      4%     8%      0
JNJ       110      4%     4%      0
WMT        83      4%     2%      0
PG         89      3%     9%      0
NVDA       61      3%     2%      0
JPM       310      3%     3%      0
AMZN       65      3%     5%      0
CVX       195      3%    31%      0
BAC       174      2%     1%      0
XOM       134      1%     3%      0
UNH        72      1%     6%      0
MSFT       87      1%     1%      0
BA        120      1%     4%      0
GS        235      0%     0%      0


In [3]:
# Finds PFE's parsed file without hardcoding the date
pfe_path = next(Path("../data/processed").glob("PFE_10-K_*.json"))
doc = json.loads(pfe_path.read_text(encoding="utf-8"))

small = [
	b for b in doc["blocks"]
	# Keeps tables with at most three lines: header, separator, and one data row
  	if b["type"] == "table" and len(b["content"].splitlines()) <= 3   # one data row
]
print(f"{len(small)} one-row tables\n")

for b in small[:15]:
	print(b["content"].splitlines()[2][:150])   # line 2 = the data row

10 one-row tables

| ☒ | ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934 |
| (State or other jurisdiction of incorporation or organization) | (I.R.S. Employer Identification Number) |
| Portions of the Proxy Statement for the 2026 Annual Meeting of Shareholders | Part III |
| A decrease of 2% compared to 2024 | A decrease of 8% compared to 2024 |
| A decrease of 3% compared to 2024 | An increase of 4% compared to 2024 |
| Expected annual rate of return on plan assets (a) | 50 basis point decline | $87 |
| Discount rate | 10 basis point decline | $5 | $201 |
| February 26, 2026 |
| Amortization expense | 4,677 | 4,087 | 3,723 | 2,775 | 2,732 |
| February 26, 2026 |


In [4]:
# Shows headerless tables for any company.
def show_no_header(ticker, n=5, max_lines=5):
	"""Print the first n tables without a header for one company."""
	path = next(Path("../data/processed").glob(f"{ticker}_10-K_*.json"))
	doc = json.loads(path.read_text(encoding="utf-8"))
	bad = [
		b for b in doc["blocks"]
		if b["type"] == "table" and not re.search(r"\w", b["content"].splitlines()[0])
	]
	print(f"{ticker}: {len(bad)} tables without a header\n")
	for b in bad[:n]:
		print("CONTEXT:", b["context"][:150])
		print("\n".join(b["content"].splitlines()[:max_lines]))
		print()

show_no_header("PFE")
show_no_header("JNJ")

PFE: 5 tables without a header

CONTEXT: See the Item 1. Business—Patents and Other Intellectual Property Rights section for information regarding the expiration of various patent rights, Not
|  |  |  |  |  |  |
| --- | --- | --- | --- | --- | --- |
| Costs and expenses follow - Year Ended December 31, | % Change |  |  |  |  |
| Costs and expenses follow - (MILLIONS) | 2025 | 2024 | 2023 | 25/24 | 24/23 |
| Costs and expenses follow - Cost of sales | 16,067 | 17,851 | 24,954 | (10) | (28) |

CONTEXT: Included in net unrealized (gains)/losses are observable price changes on equity securities without readily determinable fair values. As of December 3
|  |  |  |
| --- | --- | --- |
| Short-term borrowings include - As of December 31, |  |  |
| Short-term borrowings include - (MILLIONS) | 2025 | 2024 |
| Short-term borrowings include - Commercial paper, principal amount (a) | — | 2,453 |

CONTEXT: We determine if an arrangement is a lease at inception of the contract and we perform the lea